## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-10-08-50-00 +0000,wsj,Better Risk Appetite Weighs on Dollar; Treasur...,https://www.wsj.com/articles/dollar-weakens-ah...
1,2026-11-10-08-50-00 +0000,wsj,Volkswagen Backer Porsche SE Urges Faster Cost...,https://www.wsj.com/business/earnings/volkswag...
2,2026-11-10-08-44-59 +0000,bbc,Hundreds more offenders leaving jail on Tuesday,https://www.bbc.co.uk/news/articles/c51kdk8z7y...
3,2026-11-10-08-39-05 +0000,bbc,DR Congo sentences 66 soldiers to death for be...,https://www.bbc.co.uk/news/articles/cmreye1q3d...
4,2026-11-10-08-38-00 +0000,wsj,Chip Maker Infineon Lifts AI Sales Goal on Str...,https://www.wsj.com/business/earnings/chip-mak...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2695/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
62,trump,43
65,nvidia,43
88,huang,30
70,spy,24
87,jensen,23


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
68,2026-11-10-02-13-11 +0000,nyt,Nvidia CEO Jensen Huang Told Trump He Did Not ...,https://www.nytimes.com/2026/11/09/us/politics...,236
156,2026-11-09-22-16-39 +0000,nypost,Nvidia boss Jensen Huang had ‘absolutely no id...,https://nypost.com/2026/11/09/us-news/nvidia-b...,224
212,2026-11-09-20-39-13 +0000,nypost,Trader warned of Nvidia CEO Jensen Huang’s ‘un...,https://nypost.com/2026/11/09/business/trader-...,210
239,2026-11-09-19-47-46 +0000,nypost,Nvidia CEO Jensen Huang has no obvious success...,https://nypost.com/2026/11/09/business/nvidia-...,199
238,2026-11-09-19-53-19 +0000,nypost,"Bill Ackman, Elon Musk back Nvidia CEO Jensen ...",https://nypost.com/2026/11/09/business/bill-ac...,193


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
68,236,2026-11-10-02-13-11 +0000,nyt,Nvidia CEO Jensen Huang Told Trump He Did Not ...,https://www.nytimes.com/2026/11/09/us/politics...
166,74,2026-11-09-22-06-00 +0000,wsj,Veteran GOP lawmaker Steve Scalise is going he...,https://www.wsj.com/politics/why-steve-scalise...
241,73,2026-11-09-19-38-13 +0000,nypost,Rep. Thomas Massie reintroduces impeachment ar...,https://nypost.com/2026/11/09/us-news/rep-thom...
55,70,2026-11-10-03-28-00 +0000,wsj,California voters rejected a proposed billiona...,https://www.wsj.com/politics/elections/califor...
201,61,2026-11-09-21-11-22 +0000,chitri,Supreme Court clears the way for border wall c...,https://www.chicagotribune.com/2026/11/09/supr...
165,55,2026-11-09-22-06-57 +0000,nypost,Trump holds White House briefing with UFC’s Da...,https://nypost.com/2026/11/09/us-news/trump-ho...
411,50,2026-11-09-09-04-21 +0000,wsj,"Stock Market News, November 9, 2026: Nvidia St...",https://www.wsj.com/livecoverage/stock-market-...
374,49,2026-11-09-11-48-00 +0000,wsj,This week will pose two major tests for Presid...,https://www.wsj.com/politics/policy/nvidias-je...
90,43,2026-11-10-00-53-56 +0000,nypost,Saylor Hayes’ cause of death revealed months a...,https://nypost.com/2026/11/09/us-news/saylor-h...
239,42,2026-11-09-19-47-46 +0000,nypost,Nvidia CEO Jensen Huang has no obvious success...,https://nypost.com/2026/11/09/business/nvidia-...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
